# FPO Training on Gymnasium Ant-v5 (Single Notebook)
This notebook contains the entire FPO implementation and training loop inline. No GitHub repo cloning is required.

In [ ]:
# Install required packages
!pip install -q "gymnasium[mujoco]" optax matplotlib pandas jax jaxlib

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'egl'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

import gymnasium as gym
import jax
import jax.numpy as jnp
import numpy as np
import optax
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

print('Imports ready')

In [ ]:
import dataclasses
from dataclasses import dataclass
from typing import Tuple, Any

@jax.tree_util.register_pytree_node_class
@dataclass
class NormalDistribution:
    loc: jnp.ndarray
    scale: jnp.ndarray

    def sample(self, seed: jnp.ndarray) -> jnp.ndarray:
        return jax.random.normal(seed, shape=self.loc.shape) * self.scale + self.loc

    def log_prob(self, x: jnp.ndarray) -> jnp.ndarray:
        log_unnormalized = -0.5 * jnp.square(x / self.scale - self.loc / self.scale)
        log_normalization = 0.5 * jnp.log(2.0 * jnp.pi) + jnp.log(self.scale)
        return log_unnormalized - log_normalization

    def entropy(self) -> jnp.ndarray:
        log_normalization = 0.5 * jnp.log(2.0 * jnp.pi) + jnp.log(self.scale)
        return (0.5 + log_normalization) * jnp.ones_like(self.loc)

MlpWeights = tuple[tuple[jnp.ndarray, jnp.ndarray], ...]


def mlp_init(prng: jnp.ndarray, dims: tuple[int, ...]) -> MlpWeights:
    prngs = jax.random.split(prng, len(dims) - 1)
    init_fn = jax.nn.initializers.lecun_uniform()
    return tuple(
        (init_fn(p, (in_dim, out_dim)), jnp.zeros((out_dim,)))
        for p, in_dim, out_dim in zip(prngs, dims[:-1], dims[1:])
    )


def value_mlp_fwd(weights: MlpWeights, x: jnp.ndarray) -> jnp.ndarray:
    for linear, bias in weights[:-1]:
        x = jnp.einsum('...i,ij->...j', x, linear) + bias
        x = jax.nn.silu(x)
    linear, bias = weights[-1]
    x = jnp.einsum('...i,ij->...j', x, linear) + bias
    return jnp.squeeze(x, axis=-1)


def flow_mlp_fwd(weights: MlpWeights, *inputs: jnp.ndarray) -> jnp.ndarray:
    x = jnp.concatenate(inputs, axis=-1)
    for linear, bias in weights[:-1]:
        x = jnp.einsum('...i,ij->...j', x, linear) + bias
        x = jax.nn.silu(x)
    linear, bias = weights[-1]
    return jnp.einsum('...i,ij->...j', x, linear) + bias


@jax.tree_util.register_pytree_node_class
@dataclass
class RunningStats:
    count: jnp.ndarray
    mean: jnp.ndarray
    var_sum: jnp.ndarray
    std: jnp.ndarray

    @staticmethod
    def init(shape: tuple[int, ...]) -> 'RunningStats':
        return RunningStats(
            count=jnp.zeros((), dtype=jnp.float32),
            mean=jnp.zeros(shape, dtype=jnp.float32),
            var_sum=jnp.zeros(shape, dtype=jnp.float32),
            std=jnp.ones(shape, dtype=jnp.float32),
        )

    def update(self, x: jnp.ndarray) -> 'RunningStats':
        batch_ndims = x.ndim - self.mean.ndim
        assert x.shape[batch_ndims:] == self.mean.shape
        new_count = self.count + np.prod(x.shape[:batch_ndims])
        diff_to_old_mean = x - self.mean
        new_mean = self.mean + jnp.sum(diff_to_old_mean, axis=tuple(range(batch_ndims))) / new_count
        new_var_sum = self.var_sum + jnp.sum(diff_to_old_mean * (x - new_mean), axis=tuple(range(batch_ndims)))
        var_clipped = jnp.clip(new_var_sum / new_count, 1e-12, 1e12)
        return RunningStats(
            count=new_count,
            mean=new_mean,
            var_sum=new_var_sum,
            std=jnp.sqrt(var_clipped),
        )


@jax.tree_util.register_pytree_node_class
@dataclass
class TransitionStruct:
    obs: jnp.ndarray
    next_obs: jnp.ndarray
    action: jnp.ndarray
    action_info: Any
    reward: jnp.ndarray
    truncation: jnp.ndarray
    discount: jnp.ndarray

    def tree_flatten(self):
        children = (self.obs, self.next_obs, self.action, self.action_info, self.reward, self.truncation, self.discount)
        return children, None

    @classmethod
    def tree_unflatten(cls, aux, children):
        return cls(*children)

    def prepare_minibatches(self, prng: jnp.ndarray, num_minibatches: int, minibatch_size: int) -> 'TransitionStruct':
        (T, num_envs, _) = self.obs.shape
        subseq_count = num_minibatches * minibatch_size
        subseq_length = T * num_envs // subseq_count
        shuffle_indices = jax.random.permutation(prng, subseq_count)

        def prepare_batch(x: jnp.ndarray) -> jnp.ndarray:
            suffix = x.shape[2:]
            x = x.swapaxes(0, 1)
            x = x.reshape((-1, subseq_length) + suffix)
            x = x[shuffle_indices, ...]
            x = x.reshape((num_minibatches, minibatch_size, subseq_length) + suffix)
            x = x.swapaxes(1, 2)
            return x

        return TransitionStruct(
            obs=prepare_batch(self.obs),
            next_obs=prepare_batch(self.next_obs),
            action=prepare_batch(self.action),
            action_info=jax.tree_map(prepare_batch, self.action_info),
            reward=prepare_batch(self.reward),
            truncation=prepare_batch(self.truncation),
            discount=prepare_batch(self.discount),
        )


def compute_gae(truncation: jnp.ndarray, discount: jnp.ndarray, rewards: jnp.ndarray, values: jnp.ndarray, bootstrap_value: jnp.ndarray, gae_lambda: float):
    trunc_mask = 1 - truncation
    values_t_plus_1 = jnp.concatenate([values[1:], bootstrap_value], axis=0)
    deltas = rewards + discount * values_t_plus_1 - values
    deltas = deltas * trunc_mask
    accum_scale = discount * gae_lambda * trunc_mask

    def compute_vs_minus_v_xs(carry, x):
        acc = carry
        delta_t, accum_scale_t = x
        acc = delta_t + accum_scale_t * acc
        return acc, acc

    _, vs_minus_v_xs = jax.lax.scan(
        compute_vs_minus_v_xs,
        init=jnp.zeros_like(bootstrap_value.squeeze(axis=0)),
        xs=(deltas, accum_scale),
        reverse=True,
    )
    gae_values = vs_minus_v_xs + values
    gae_values_t_plus_1 = jnp.concatenate([gae_values[1:], bootstrap_value], axis=0)
    advantages = rewards + discount * gae_values_t_plus_1 - values
    advantages = advantages * trunc_mask
    return gae_values, advantages

In [ ]:
@jax.tree_util.register_pytree_node_class
@dataclass
class FpoConfig:
    flow_steps: int = 10
    output_mode: str = 'u_but_supervise_as_eps'
    timestep_embed_dim: int = 8
    n_samples_per_action: int = 8
    average_losses_before_exp: bool = True
    discretize_t_for_training: bool = True
    feather_std: float = 0.0
    policy_mlp_output_scale: float = 0.25
    loss_mode: str = 'fpo'
    final_steps_only: bool = False
    sde_sigma: float = 0.0
    clipping_epsilon: float = 0.05
    batch_size: int = 256
    discounting: float = 0.995
    episode_length: int = 1000
    learning_rate: float = 3e-4
    normalize_observations: bool = True
    num_envs: int = 8
    num_evals: int = 10
    num_minibatches: int = 4
    num_timesteps: int = 153600
    num_updates_per_batch: int = 2
    reward_scaling: float = 10.0
    unroll_length: int = 30
    gae_lambda: float = 0.95
    normalize_advantage: bool = True

    @property
    def iterations_per_env(self) -> int:
        return (self.num_minibatches * self.batch_size * self.unroll_length) // self.num_envs


@jax.tree_util.register_pytree_node_class
@dataclass
class FpoParams:
    policy: MlpWeights
    value: MlpWeights

    def tree_flatten(self):
        return ((self.policy, self.value), None)

    @classmethod
    def tree_unflatten(cls, aux, children):
        return cls(*children)


@jax.tree_util.register_pytree_node_class
@dataclass
class FpoActionInfo:
    loss_eps: jnp.ndarray
    loss_t: jnp.ndarray
    initial_cfm_loss: jnp.ndarray

    def tree_flatten(self):
        return ((self.loss_eps, self.loss_t, self.initial_cfm_loss), None)

    @classmethod
    def tree_unflatten(cls, aux, children):
        return cls(*children)


@jax.tree_util.register_pytree_node_class
@dataclass
class FlowSchedule:
    t_current: jnp.ndarray
    t_next: jnp.ndarray


@jax.tree_util.register_pytree_node_class
@dataclass
class FpoState:
    env: Any
    config: FpoConfig
    params: FpoParams
    obs_stats: RunningStats
    opt_state: Any
    prng: jnp.ndarray
    steps: jnp.ndarray

    def tree_flatten(self):
        children = (self.params, self.obs_stats, self.opt_state, self.prng, self.steps)
        aux = (self.env, self.config)
        return children, aux

    @classmethod
    def tree_unflatten(cls, aux, children):
        env, config = aux
        params, obs_stats, opt_state, prng, steps = children
        return cls(env=env, config=config, params=params, obs_stats=obs_stats, opt_state=opt_state, prng=prng, steps=steps)

    @staticmethod
    def init(prng: jnp.ndarray, env: Any, config: FpoConfig) -> 'FpoState':
        obs_size = env.observation_size
        action_size = env.action_size
        prng0, prng1, prng2 = jax.random.split(prng, num=3)
        actor_net = mlp_init(prng0, (obs_size + action_size + config.timestep_embed_dim, 32, 32, 32, 32, action_size))
        critic_net = mlp_init(prng1, (obs_size, 256, 256, 256, 256, 256, 1))
        params = FpoParams(actor_net, critic_net)
        obs_stats = RunningStats.init((obs_size,))
        opt = optax.scale_by_adam()
        opt_state = opt.init(params)
        return FpoState(env=env, config=config, params=params, obs_stats=obs_stats, opt_state=opt_state, prng=prng2, steps=jnp.zeros((), dtype=jnp.int32))

    def get_schedule(self) -> FlowSchedule:
        full_t_path = jnp.linspace(1.0, 0.0, self.config.flow_steps + 1)
        return FlowSchedule(t_current=full_t_path[:-1], t_next=full_t_path[1:])

    def embed_timestep(self, t: jnp.ndarray) -> jnp.ndarray:
        freqs = 2 ** jnp.arange(self.config.timestep_embed_dim // 2)
        scaled_t = t * freqs
        return jnp.concatenate([jnp.cos(scaled_t), jnp.sin(scaled_t)], axis=-1)

    def _compute_cfm_loss(self, obs_norm: jnp.ndarray, action: jnp.ndarray, eps: jnp.ndarray, t: jnp.ndarray) -> jnp.ndarray:
        (*batch_dims, action_dim) = action.shape
        sample_shape = (*batch_dims, self.config.n_samples_per_action)
        x_t = t * eps + (1.0 - t) * action[..., None, :]
        network_pred = flow_mlp_fwd(
            self.params.policy,
            jnp.broadcast_to(obs_norm[..., None, :], (*sample_shape, self.env.observation_size)),
            x_t,
            jnp.broadcast_to(self.embed_timestep(t), (*sample_shape, self.config.timestep_embed_dim)),
        ) * self.config.policy_mlp_output_scale
        if self.config.output_mode == 'u':
            velocity_pred = network_pred
            velocity_gt = eps - action[..., None, :]
            out = jnp.mean((velocity_pred - velocity_gt) ** 2, axis=-1)
        else:
            velocity_pred = network_pred
            x0_pred = x_t - t * velocity_pred
            x1_pred = x0_pred + velocity_pred
            out = jnp.mean((eps - x1_pred) ** 2, axis=-1)
        return out

    def _compute_denoising_log_likelihood(self, obs_norm: jnp.ndarray, x_t_path: jnp.ndarray) -> jnp.ndarray:
        (*batch_dims, total_states, action_dim) = x_t_path.shape
        schedule = self.get_schedule()
        x_t = x_t_path[..., :-1, :]
        x_t_next = x_t_path[..., 1:, :]
        dt = schedule.t_next - schedule.t_current
        velocity_pred = flow_mlp_fwd(
            self.params.policy,
            jnp.broadcast_to(obs_norm[..., None, :], (*batch_dims, self.env.observation_size)),
            x_t,
            jnp.broadcast_to(self.embed_timestep(schedule.t_current[None]), (*batch_dims, self.config.flow_steps, self.config.timestep_embed_dim)),
        ) * self.config.policy_mlp_output_scale
        expected_x_t_next = x_t + dt[None, :, None] * velocity_pred
        realized_noise = (x_t_next - expected_x_t_next) / (self.config.sde_sigma + 1e-6)[None, :, None]
        return -0.5 * jnp.sum(realized_noise ** 2, axis=-1) - 0.5 * action_dim * jnp.log(2 * jnp.pi)

    def sample_action(self, obs: jnp.ndarray, prng: jnp.ndarray, deterministic: bool) -> tuple[jnp.ndarray, FpoActionInfo]:
        if self.config.normalize_observations:
            obs_norm = (obs - self.obs_stats.mean) / self.obs_stats.std
        else:
            obs_norm = obs
        (*batch_dims, obs_dim) = obs.shape
        schedule = self.get_schedule()

        def euler_step(x_t, inputs):
            schedule_t, noise = inputs
            dt = schedule_t.t_next - schedule_t.t_current
            velocity = flow_mlp_fwd(
                self.params.policy,
                obs_norm,
                x_t,
                jnp.broadcast_to(self.embed_timestep(schedule_t.t_current[None]), (*batch_dims, self.config.timestep_embed_dim)),
            ) * self.config.policy_mlp_output_scale
            x_t_next = x_t + dt * velocity + self.config.sde_sigma * noise
            return x_t_next, x_t

        prng_sample, prng_loss, prng_feather, prng_noise = jax.random.split(prng, num=4)
        noise_path = jax.random.normal(prng_noise, (self.config.flow_steps, *batch_dims, self.env.action_size))
        init_x = jax.random.normal(prng_sample, (*batch_dims, self.env.action_size))

        x0, x_t_path = jax.lax.scan(
            euler_step,
            init_x,
            (schedule, noise_path),
        )

        if not deterministic:
            perturb = jax.random.normal(prng_feather, (*batch_dims, self.env.action_size)) * self.config.feather_std
            x0 = x0 + perturb

        prng_eps, prng_t = jax.random.split(prng_loss)
        eps = jax.random.normal(prng_eps, (*batch_dims, self.config.n_samples_per_action, self.env.action_size))
        if self.config.discretize_t_for_training:
            idx = jax.random.randint(prng_t, (*batch_dims, 1), 0, self.config.flow_steps)
            t = schedule.t_current[idx]
        else:
            t = jax.random.uniform(prng_t, (*batch_dims, 1))
        initial_cfm_loss = self._compute_cfm_loss(obs_norm, x0, eps=eps, t=t)
        action_info = FpoActionInfo(loss_eps=eps, loss_t=t, initial_cfm_loss=initial_cfm_loss)
        return x0, action_info

    def training_step(self, transitions: TransitionStruct) -> tuple['FpoState', dict[str, jnp.ndarray]]:
        if self.config.normalize_observations:
            self = dataclasses.replace(self, obs_stats=self.obs_stats.update(transitions.obs))

        def step_batch(state, _):
            prng_batch = jax.random.fold_in(state.prng, state.steps)
            state, metrics = jax.lax.scan(
                lambda st, batch: st._step_minibatch(st, batch, jax.random.fold_in(prng_batch, 0)),
                state,
                transitions.prepare_minibatches(prng_batch, self.config.num_minibatches, self.config.batch_size),
            )
            return state, metrics

        self, metrics = jax.lax.scan(step_batch, self, jnp.arange(self.config.num_updates_per_batch))
        return self, metrics

    def _step_minibatch(self, state: 'FpoState', transitions: TransitionStruct, prng: jnp.ndarray) -> tuple['FpoState', dict[str, jnp.ndarray]]:
        def loss_fn(params):
            s = dataclasses.replace(state, params=params)
            return s._compute_fpo_loss(transitions, prng)

        (loss, metrics), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
        opt = optax.scale_by_adam()
        updates, new_opt_state = opt.update(grads, state.opt_state)
        updates = jax.tree_map(lambda x: -self.config.learning_rate * x, updates)
        new_params = jax.tree_map(lambda p, u: p + u, state.params, updates)
        return dataclasses.replace(state, params=new_params, opt_state=new_opt_state, steps=state.steps + 1), metrics

    def _compute_fpo_loss(self, transitions: TransitionStruct, prng: jnp.ndarray) -> tuple[jnp.ndarray, dict[str, jnp.ndarray]]:
        del prng
        if self.config.normalize_observations:
            obs_norm = (transitions.obs - self.obs_stats.mean) / self.obs_stats.std
        else:
            obs_norm = transitions.obs
        value_pred = value_mlp_fwd(self.params.value, obs_norm)
        bootstrap_obs_norm = (transitions.next_obs[-1:, :, :] - self.obs_stats.mean) / self.obs_stats.std
        bootstrap_value = value_mlp_fwd(self.params.value, bootstrap_obs_norm)
        gae_vs, gae_advantages = compute_gae(
            truncation=transitions.truncation,
            discount=transitions.discount * self.config.discounting,
            rewards=transitions.reward * self.config.reward_scaling,
            values=value_pred,
            bootstrap_value=bootstrap_value,
            gae_lambda=self.config.gae_lambda,
        )
        metrics = {
            'advantages_mean': jnp.mean(gae_advantages),
            'advantages_std': jnp.std(gae_advantages),
            'advantages_min': jnp.min(gae_advantages),
            'advantages_max': jnp.max(gae_advantages),
        }
        if self.config.normalize_advantage:
            gae_advantages = (gae_advantages - gae_advantages.mean()) / (gae_advantages.std() + 1e-8)
        cfm_loss = self._compute_cfm_loss(
            obs_norm,
            transitions.action,
            eps=transitions.action_info.loss_eps,
            t=transitions.action_info.loss_t,
        )
        if self.config.average_losses_before_exp:
            rho_s = jnp.exp(
                jnp.mean(transitions.action_info.initial_cfm_loss, axis=-1, keepdims=True)
                - jnp.mean(cfm_loss, axis=-1, keepdims=True)
            )
        else:
            rho_s = jnp.exp(jnp.clip(transitions.action_info.initial_cfm_loss - cfm_loss, -3.0, 3.0))
        surrogate_loss1 = rho_s * gae_advantages[..., None]
        surrogate_loss2 = jnp.clip(rho_s, 1 - self.config.clipping_epsilon, 1 + self.config.clipping_epsilon) * gae_advantages[..., None]
        policy_loss = -jnp.mean(jnp.minimum(surrogate_loss1, surrogate_loss2))
        v_error = (gae_vs - value_pred) * (1 - transitions.truncation)
        metrics.update({
            'clipped_ratio_mean': jnp.mean(jnp.abs(rho_s - 1.0) > self.config.clipping_epsilon),
            'policy_ratio_mean': jnp.mean(rho_s),
            'policy_ratio_min': jnp.min(rho_s),
            'policy_ratio_max': jnp.max(rho_s),
            'policy_loss': policy_loss,
            'value_mean': jnp.mean(value_pred),
            'value_std': jnp.std(value_pred),
            'value_min': jnp.min(value_pred),
            'value_max': jnp.max(value_pred),
            'value_target_mean': jnp.mean(gae_vs),
            'value_error_mean': jnp.mean(v_error),
            'v_loss': jnp.mean(v_error ** 2) * self.config.value_loss_coeff,
        })
        total_loss = policy_loss + metrics['v_loss']
        return total_loss, metrics


In [ ]:
class GymAntEnv:
    def __init__(self, env_name: str, num_envs: int):
        self.env = gym.vector.SyncVectorEnv([lambda: gym.make(env_name) for _ in range(num_envs)])
        self.action_space = self.env.single_action_space
        self.observation_space = self.env.single_observation_space
        self.observation_size = int(np.prod(self.observation_space.shape))
        self.action_size = int(np.prod(self.action_space.shape))

    def reset(self) -> np.ndarray:
        obs, _ = self.env.reset()
        return np.asarray(obs, dtype=np.float32)

    def step(self, action: np.ndarray):
        action = np.asarray(action, dtype=np.float32)
        action = np.clip(action, self.action_space.low, self.action_space.high)
        obs, reward, terminated, truncated, infos = self.env.step(action)
        return (
            np.asarray(obs, dtype=np.float32),
            np.asarray(reward, dtype=np.float32),
            np.asarray(terminated, dtype=np.bool_),
            np.asarray(truncated, dtype=np.bool_),
            infos,
        )


def extract_episode_stats(rewards: np.ndarray, truncation: np.ndarray, discount: np.ndarray):
    episode_rewards = []
    episode_lengths = []
    T, B = rewards.shape
    for b in range(B):
        current_reward = 0.0
        current_length = 0
        for t in range(T):
            current_reward += float(rewards[t, b])
            current_length += 1
            if truncation[t, b] > 0.5 or discount[t, b] == 0.0:
                episode_rewards.append(current_reward)
                episode_lengths.append(current_length)
                current_reward = 0.0
                current_length = 0
    return np.array(episode_rewards, dtype=np.float32), np.array(episode_lengths, dtype=np.int32)


def collect_transitions(env: GymAntEnv, agent_state: FpoState, prng: jnp.ndarray, config: FpoConfig):
    obs = jnp.asarray(env.reset(), dtype=jnp.float32)
    obs_seq = []
    next_obs_seq = []
    action_seq = []
    reward_seq = []
    truncation_seq = []
    discount_seq = []
    action_info_seq = []
    for _ in range(config.iterations_per_env):
        prng, prng_step = jax.random.split(prng)
        action, action_info = agent_state.sample_action(obs, prng_step, deterministic=False)
        next_obs_np, reward_np, terminated_np, truncated_np, _ = env.step(np.asarray(action))
        next_obs = jnp.asarray(next_obs_np, dtype=jnp.float32)
        reward = jnp.asarray(reward_np, dtype=jnp.float32)
        truncation = jnp.asarray(truncated_np, dtype=jnp.float32)
        discount = 1.0 - jnp.asarray(terminated_np, dtype=jnp.float32)
        obs_seq.append(obs)
        next_obs_seq.append(next_obs)
        action_seq.append(action)
        reward_seq.append(reward)
        truncation_seq.append(truncation)
        discount_seq.append(discount)
        action_info_seq.append(action_info)
        obs = next_obs
    transitions = TransitionStruct(
        obs=jnp.stack(obs_seq),
        next_obs=jnp.stack(next_obs_seq),
        action=jnp.stack(action_seq),
        action_info=jax.tree_map(lambda *xs: jnp.stack(xs), *action_info_seq),
        reward=jnp.stack(reward_seq),
        truncation=jnp.stack(truncation_seq),
        discount=jnp.stack(discount_seq),
    )
    return transitions, prng


def run_training():
    env_name = 'Ant-v5'
    num_envs = 8
    seed = 42
    config = FpoConfig(num_envs=num_envs)
    env = GymAntEnv(env_name, num_envs)
    prng = jax.random.PRNGKey(seed)
    agent_state = FpoState.init(prng, env, config)
    prng = jax.random.split(prng, num=2)[1]
    outer_iters = int(config.num_timesteps // (config.iterations_per_env * num_envs))
    history = []

    for i in range(outer_iters):
        transitions, prng = collect_transitions(env, agent_state, prng, config)
        agent_state, metrics = agent_state.training_step(transitions)
        reward_np = np.asarray(transitions.reward)
        truncation_np = np.asarray(transitions.truncation)
        discount_np = np.asarray(transitions.discount)
        episode_rewards, episode_lengths = extract_episode_stats(reward_np, truncation_np, discount_np)
        mean_reward = float(np.mean(reward_np))
        mean_episode_reward = float(np.mean(episode_rewards)) if episode_rewards.size else mean_reward
        mean_episode_length = float(np.mean(episode_lengths)) if episode_lengths.size else 0.0
        row = {
            'step': i,
            'mean_reward': mean_reward,
            'mean_episode_reward': mean_episode_reward,
            'mean_episode_length': mean_episode_length,
            'num_episodes': int(episode_rewards.size),
        }
        history.append(row)
        print(f'Iter {i+1}/{outer_iters} | mean_reward={mean_reward:.3f} | mean_episode_reward={mean_episode_reward:.3f} | episodes={row["num_episodes"]}')
        if (i + 1) % 1 == 0 or i == outer_iters - 1:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, 5))
            ax.plot([h['step'] for h in history], [h['mean_reward'] for h in history], label='mean_reward')
            ax.plot([h['step'] for h in history], [h['mean_episode_reward'] for h in history], label='mean_episode_reward')
            ax.set_title(f'{env_name} FPO Training (Iter {i+1})')
            ax.set_xlabel('outer iteration')
            ax.set_ylabel('reward')
            ax.legend()
            display(fig)
            plt.close(fig)
    df = pd.DataFrame(history)
    df.to_csv('fpo_ant_training_metrics.csv', index=False)
    print('Training complete. Metrics saved to fpo_ant_training_metrics.csv')
    return agent_state, df

In [ ]:
agent_state, history_df = run_training()
print(history_df)